# Build feature-vector snapshots from trades

`fiml` accepts an already-loaded, globally ordered trade DataFrame. File loading stays in pandas.

In [1]:
import numpy as np
import pandas as pd
import fiml

In [2]:
trades = pd.read_csv("trades.csv")
trades

,symbol,ts,price,volume
0,BTCUSDT,1783950118926,1.0,3
1,BTCUSDT,1783950118936,1.1,4
2,BTCUSDT,1783950118946,1.2,3
3,BTCUSDT,1783950118956,1.0,2


In [3]:
feature_set = (fiml.FeatureSet()
    .obv_timed("BTCUSDT", aggregation="1ms", windows=["60s"])
    .trade_count_timed("BTCUSDT", aggregation="1ms", window="60s")
    .sma("BTCUSDT", [2], source="trade_price")
    .ema("BTCUSDT", [2], source="trade_price")
    .day_of_week())
extractor = fiml.FeatureExtractor(feature_set, output_dtype=np.float32)

In [4]:
features = extractor.compute_features(trades)
features

,symbol,ts,BTCUSDT:trade:obv_timed:1ms:60000ms,BTCUSDT:trade:count_timed:1ms:60000ms,BTCUSDT:trade_price:sma:2,BTCUSDT:trade_price:ema:2,clock:day_of_week
0,BTCUSDT,1783950118926,0.0,1.0,0.50,1.000000,1.0
1,BTCUSDT,1783950118936,4.0,2.0,1.05,1.066667,1.0
2,BTCUSDT,1783950118946,7.0,3.0,1.15,1.155556,1.0
3,BTCUSDT,1783950118956,5.0,4.0,1.10,1.051852,1.0


In [5]:
assert features.index.equals(trades.index)
assert features["symbol"].equals(trades["symbol"])
assert features["ts"].equals(trades["ts"])
assert list(features.columns[2:]) == extractor.feature_names()
assert all(dtype == np.float32 for dtype in features.dtypes.iloc[2:])
assert features.shape == (len(trades), extractor.n_features() + 2)
assert not features[["BTCUSDT:trade_price:sma:2", "BTCUSDT:trade_price:ema:2"]].isna().any().any()